# 🎬 YouTube DataNexus Baseline (Data Pipeline)
이 노트북은 `참고/` 폴더의 설계 문서를 기반으로 작성된 파이프라인의 핵심 코드(백엔드)입니다.
스트림릿(Streamlit) 없이 코어 로직(다운로드 -> STT -> Vector DB)을 실험하고 테스트할 수 있도록 작성되었습니다.

In [1]:
# 필요한 패키지 설치
!pip install yt-dlp openai-whisper imageio-ffmpeg langchain langchain-openai faiss-cpu

Defaulting to user installation because normal site-packages is not writeable


## 1. 초기 설정 및 유튜브 다운로드 (`yt-dlp`)

In [2]:
import os
import yt_dlp
import whisper
import json

# 캐시 폴더 생성
CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# 테스트용 유튜브 링크
YOUTUBE_URL = "https://youtu.be/Ot4dcoBOHWw"

def get_video_id(url):
    import re
    match = re.search(r"(?:v=|youtu\.be\/)([0-9A-Za-z_-]{11})", url)
    return match.group(1) if match else None

video_id = get_video_id(YOUTUBE_URL)
audio_path = os.path.join(CACHE_DIR, f"{video_id}.mp3")

# MP3 다운로드 함수 (캐싱 적용)
if not os.path.exists(audio_path):
    print("🎵 오디오 다운로드 중...")
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'outtmpl': os.path.join(CACHE_DIR, f"{video_id}.%(ext)s"),
        'quiet': False,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YOUTUBE_URL])
    print("✅ 다운로드 완료!")
else:
    print("✅ 캐시된 오디오 파일을 사용합니다.")

✅ 캐시된 오디오 파일을 사용합니다.


## 2. Whisper를 이용한 STT (Speech-to-Text)

In [3]:
# STT 변환 및 JSON 저장
transcript_path = os.path.join(CACHE_DIR, f"{video_id}.json")

if not os.path.exists(transcript_path):
    print("🗣️ Whisper AI 모델 로딩 중...")
    model = whisper.load_model("base")
    print("🗣️ 음성 변환(STT) 진행 중... (시간이 소요될 수 있습니다)")
    result = model.transcribe(audio_path)
    
    with open(transcript_path, "w", encoding="utf-8") as f:
        json.dump(result["segments"], f, ensure_ascii=False, indent=4)
        
    print("✨ STT 변환 완료!")
    segments = result["segments"]
else:
    print("✅ 캐시된 STT 결과(JSON)를 사용합니다.")
    with open(transcript_path, "r", encoding="utf-8") as f:
        segments = json.load(f)

# 첫 5개의 구간만 출력
for seg in segments[:5]:
    print(f"[{seg['start']:.2f}s - {seg['end']:.2f}s] {seg['text']}")

✅ 캐시된 STT 결과(JSON)를 사용합니다.
[0.00s - 1.00s]  아...
[8.50s - 10.50s]  알람이 특이한 오늘의 집
[24.00s - 25.00s]  어... 어...
[25.00s - 27.00s]  오늘의 주인공 박시르다
[30.00s - 32.00s]  싫어, 싫어, 싫어?


## 3. RAG 파이프라인 준비 (Vector DB 연동 예시)
`Design Document.txt`의 다음 단계를 위해 텍스트 청크를 나누고 Vector DB(FAISS 등)에 저장할 수 있는 기반 코드입니다.

In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 병합 및 청킹
full_text = " ".join([seg['text'] for seg in segments])

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

chunks = text_splitter.split_text(full_text)
print(f"총 {len(chunks)}개의 청크(Chunk)로 나뉘었습니다.\n")

# 첫 번째 청크 확인
if chunks:
    print("🔖 첫 번째 청크 샘플:")
    print(chunks[0])

총 6개의 청크(Chunk)로 나뉘었습니다.

🔖 첫 번째 청크 샘플:
아...  알람이 특이한 오늘의 집  어... 어...  오늘의 주인공 박시르다  싫어, 싫어, 싫어?  아, 졸려  수고하셨어?  어, 출근 안 하고 싫어하는 놈까지면서?  그게 무슨 소리? 내 인시로야  그래, 또 출구도 해야지  군무닝 궁디팡팡으로 하루를 시작한다  어...  두 끼 고양이  강아지 물게  싫어, 나 출근해야 돼  좀 비켜줄래?  오오오오오  오늘도 마트니를 잘할 수 있게 해 주시오  아침부터 장난을 친다  100에도 숨숨 집이 되어버렸다  아침부터 음료수를 마시는 싫어씨  싫어 나 사녀올게  좀 자고 있어?  할 거 없으면  청소소 좀 하고  집사는 무사히 퇴근을 했다  오늘은 집에 가기 전 들릴 곳이 있다  대형마트에 들른 집사  사고 싶은 걸 찾은 모양이다  싫어와 똑닮은 고등어인형  4만 5,990원  결국 고민 끝에 사고야 한다  자연스럽게 케이크 코너로 발을 돌리는 집사  추루 90개의 36,790원  양스푼  40개의 27,990원  아직 있으니까


---
### ✅ 다음 단계
위의 코드를 통해 생성된 `chunks`를 **OpenAI 임베딩**을 통해 **Vector DB(FAISS/Chroma)** 에 저장하고 LangChain을 활용하여 Q&A 시스템(RAG)을 구현할 수 있습니다. UI는 현재 폴더에 있는 `app.py` (Streamlit)를 통해 실행할 수 있습니다.